In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 필수 라이브러리 업데이트 (for miniconda)
!apt-get update
!apt-get install -y build-essential cmake ninja-build git wget
!pip install -q plyfile

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda

import os
os.environ["PATH"] = "/usr/local/miniconda/bin:" + os.environ["PATH"]
!conda --version

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


In [ ]:
# WeatherEdit clone
!git clone --recursive https://github.com/Jumponthemoon/WeatherEdit.git
%cd WeatherEdit/General_Scene

# Submodules are handled by --recursive, no need to clone manually

In [ ]:
# g++-10 설치 (CUDA 11.6을 위해)
!apt-get install -y g++-10

# 버전 확인
!g++-10 --version

# 환경설정 세팅 (submodule은 설치 안될것임)
%cd /content/WeatherEdit/General_Scene
!conda env create -f environment.yml -n gaussian_splatting

# CUDA 11.6 설치
!conda run --no-capture-output -n gaussian_splatting conda install -c conda-forge cudatoolkit-dev=11.6 -y

# 버전 확인, release 11.6 이 나와야 함
!conda run -n gaussian_splatting nvcc --version

!conda run --no-capture-output -n gaussian_splatting pip install opencv-python

In [ ]:
!cd /content/WeatherEdit/General_Scene/submodules/diff-gaussian-rasterization && \
    git submodule update --init --recursive

# g++-10을 사용해서 Submodule build
!conda run --no-capture-output -n gaussian_splatting bash -c "\
    cd /content/WeatherEdit/General_Scene/submodules/diff-gaussian-rasterization && \
    export CXX=g++-10 && \
    export CC=gcc-10 && \
    python setup.py install 2>&1 | tail -20"

!conda run --no-capture-output -n gaussian_splatting bash -c "\
    cd /content/WeatherEdit/General_Scene/submodules/simple-knn && \
    export CXX=g++-10 && \
    export CC=gcc-10 && \
    python setup.py install 2>&1 | tail -20"

!conda run --no-capture-output -n gaussian_splatting bash -c "\
    cd /content/WeatherEdit/General_Scene/submodules/fused-ssim && \
    export CXX=g++-10 && \
    export CC=gcc-10 && \
    python setup.py install 2>&1 | tail -20"


In [ ]:
!cd /content/WeatherEdit/General_Scene

DATA_ROOT="/content/drive/MyDrive/Research/Dataset/free_dataset"
OUT_ROOT="/content/drive/MyDrive/Research/WeatherEdit/free_dataset_output"
#SCENES = ["hydrant", "pillar", "road", "sky", "stair"]
SCENES=["grass"]

for scene in SCENES :
    DATA_DIR=f"{DATA_ROOT}/{scene}"
    OUTPUT_DIR=f"{OUT_ROOT}/{scene}"

    # Train 3D Scene
    ! conda run --no-capture-output -n gaussian_splatting python -c "import torch; torch.cuda.empty_cache()"
    ! conda run --no-capture-output -n gaussian_splatting python train.py -s {DATA_DIR} -m {OUTPUT_DIR}

In [ ]:
# Weather Intense Adjust : strong

import json
from pathlib import Path

config = {
    "rain": {
        "nums": 70000,
        "scaling_mean": [0.0025, 0.05, 0.0025],
        "scaling_std": [0.002, 0.002, 0.002],
        "rotation_axis": [0, 0, 1],
        "opacity_mean": 0.62,
        "opacity_std": 0.0,
        "features_dc": 0.9,
        "velocity": [0, 0.10, 0]
    },
    "snow": {
        "nums": 35000,
        "scaling_mean": [0.005, 0.015, 0.008],
        "scaling_std": [0.002, 0.002, 0.002],
        "rotation_axis": [0, 0, 1],
        "opacity_mean": 0.95,
        "opacity_std": 0.22,
        "features_dc": 1.10,
        "velocity": [0, 0.02, 0]
    },
    "fog": {
        "nums": 900000,
        "scaling_mean": [0.06, 0.06, 0.06],
        "scaling_std": [0.01, 0.01, 0.01],
        "rotation_axis": [0, 0, 0],
        "opacity_mean": 0.015,
        "opacity_std": 0.01,
        "features_dc": 0.9,
        "velocity": [0, 0.0, 0]
    }
}

json_path = Path("/content/WeatherEdit/General_Scene/weather_config.json")

# 백업
if json_path.exists():
    backup_path = json_path.with_suffix(".json.bak")
    backup_path.write_text(json_path.read_text(encoding="utf-8"), encoding="utf-8")

# 덮어쓰기
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"saved: {json_path}")

In [ ]:
# Weather Intense Adjust : weak

import json
from pathlib import Path

config = {
    "rain": {
        "nums": 25000,
        "scaling_mean": [0.0018, 0.028, 0.0018],
        "scaling_std": [0.0015, 0.0015, 0.0015],
        "rotation_axis": [0, 0, 1],
        "opacity_mean": 0.62,
        "opacity_std": 0.0,
        "features_dc": 0.5,
        "velocity": [0, 0.06, 0]
    },
    "snow": {
        "nums": 12000,
        "scaling_mean": [0.0035, 0.008, 0.005],
        "scaling_std": [0.0015, 0.0015, 0.0015],
        "rotation_axis": [0, 0, 1],
        "opacity_mean": 0.95,
        "opacity_std": 0.18,
        "features_dc": 0.85,
        "velocity": [0, 0.014, 0]
    },
    "fog": {
        "nums": 500000,
        "scaling_mean": [0.05, 0.05, 0.05],
        "scaling_std": [0.01, 0.01, 0.01],
        "rotation_axis": [0, 0, 0],
        "opacity_mean": 0.010,
        "opacity_std": 0.007,
        "features_dc": 0.8,
        "velocity": [0, 0.0, 0]
    }
}


json_path = Path("/content/WeatherEdit/General_Scene/weather_config.json")

# 백업
if json_path.exists():
    backup_path = json_path.with_suffix(".json.bak")
    backup_path.write_text(json_path.read_text(encoding="utf-8"), encoding="utf-8")

# 덮어쓰기
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"saved: {json_path}")

In [ ]:
# Weather Intense Adjust : zero

import json
from pathlib import Path

config = {
    "rain": {
        "nums": 0,
        "scaling_mean": [0.0018, 0.028, 0.0018],
        "scaling_std": [0.0015, 0.0015, 0.0015],
        "rotation_axis": [0, 0, 1],
        "opacity_mean": 0.62,
        "opacity_std": 0.0,
        "features_dc": 0.5,
        "velocity": [0, 0.06, 0]
    },
    "snow": {
        "nums": 0,
        "scaling_mean": [0.0035, 0.008, 0.005],
        "scaling_std": [0.0015, 0.0015, 0.0015],
        "rotation_axis": [0, 0, 1],
        "opacity_mean": 0.95,
        "opacity_std": 0.18,
        "features_dc": 0.85,
        "velocity": [0, 0.014, 0]
    },
    "fog": {
        "nums": 0,
        "scaling_mean": [0.05, 0.05, 0.05],
        "scaling_std": [0.01, 0.01, 0.01],
        "rotation_axis": [0, 0, 0],
        "opacity_mean": 0.010,
        "opacity_std": 0.007,
        "features_dc": 0.8,
        "velocity": [0, 0.0, 0]
    }
}


json_path = Path("/content/WeatherEdit/General_Scene/weather_config.json")

# 백업
if json_path.exists():
    backup_path = json_path.with_suffix(".json.bak")
    backup_path.write_text(json_path.read_text(encoding="utf-8"), encoding="utf-8")

# 덮어쓰기
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"saved: {json_path}")

In [ ]:
!cd /content/WeatherEdit/General_Scene

DATA_ROOT="/content/drive/MyDrive/Research/Dataset/free_dataset"
OUT_ROOT="/content/drive/MyDrive/Research/26_Capstone2/WeatherEdit/free_dataset_output"
SCENES = ["grass", "hydrant", "pillar", "road", "sky", "stair"]
#SCENES=["grass"]

for scene in SCENES :
    DATA_DIR=f"{DATA_ROOT}/{scene}"
    OUTPUT_DIR=f"{OUT_ROOT}/{scene}"

    # Weather Synthesis
    ! conda run --no-capture-output -n gaussian_splatting python -c "import torch; torch.cuda.empty_cache()"
    ! conda run --no-capture-output -n gaussian_splatting python render.py -m {OUTPUT_DIR} --weather snow --fps 3
    ! conda run --no-capture-output -n gaussian_splatting python render.py -m {OUTPUT_DIR} --weather rain --fps 3